# 02 · Camada Silver — CineData Analytics

**Objetivo:** limpar, tipar, traduzir colunas para português e deduplicar os dados da Bronze (que **não é alterada**).

**Perfil da sujeira encontrada nos CSVs (e como cada regra abaixo a resolve):**

| Tabela | Problema real observado | Tratamento |
|---|---|---|
| `movies_info` | ~9 mil `id` duplicados; `status` em `Released/RELEASED/released/In-Production…`; datas em 3 formatos (`yyyy-MM-dd`, `dd/MM/yyyy`, `MM-dd-yyyy`) | dedupe por ingestão, normalização + mapa de tradução, parse por padrão (regex-guard) |
| `movies_financials` | `Unknown`/`Não Informado`, `$ 10000`, `USD 10000`, `10.0K`, `34.0M`, valores negativos, ids duplicados | limpeza de moeda + sufixos K/M/B, `<=0` vira NULL |
| `movies_metrics` | popularidade com vírgula decimal (`154,34`), *column shift* (texto nas colunas de nota/votos), notas na escala 0-100 (ex.: `79.97`) | cast seguro (regex), faixa 0–10 |
| `movies_reviews` | comentários nulos/vazios, notas nulas | `Sem comentário`, nota fora de 0–10 → NULL |
| `credits_and_tags` | separadores `,` `;` `|`; `N/A`, `[]`; lixo de *column shift* (notas `0.6`, caminhos `.jpg`, keywords, frases) | split por regex, whitelist de gêneros, filtros de entidade |

> **Importante sobre ANSI mode:** em compute serverless o `spark.sql.ansi.enabled` é `true` por padrão e um `cast` inválido *lança erro*. As regras de negócio exigem "valor inválido → NULL sem interromper o pipeline", então desligamos o ANSI nesta sessão **e** usamos *regex-guards* antes de cada cast (defesa em profundidade).

In [0]:
from decimal import Decimal
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalogo", "workspace", "Catálogo (Unity Catalog)")
CATALOGO = dbutils.widgets.get("catalogo")

# Cast inválido => NULL (não erro) e parse de data tolerante
for chave, valor in [("spark.sql.ansi.enabled", "false"),
                     ("spark.sql.legacy.timeParserPolicy", "CORRECTED")]:
    try:
        spark.conf.set(chave, valor)
    except Exception as e:
        print(f"[aviso] não foi possível definir {chave}: {e}")

spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Camada Silver: dados limpos, tipados e deduplicados'")

DataFrame[]

## Funções utilitárias (reutilizadas por várias tabelas)

In [0]:
def somente_numericos_validos(id_col):
    """id do filme deve ser inteiro em texto; descarta linhas lixo geradas por column shift."""
    return F.trim(F.col(id_col)).rlike(r"^\d+$")


def deduplicar_por_filme(df, chave="id_filme"):
    """Mantém UMA linha por filme: a de maior ingestion_datetime (versão mais recente).
    Desempate: a linha mais completa (mais colunas preenchidas) — duplicatas de origem costumam
    diferir só em caixa/valores ausentes."""
    colunas_dados = [c for c in df.columns if c not in (chave, "ingestion_datetime")]
    completude = sum(
        (F.when(F.col(c).isNotNull() & (F.trim(F.col(c).cast("string")) != ""), 1).otherwise(0)
         for c in colunas_dados),
        F.lit(0),
    )
    janela = Window.partitionBy(chave).orderBy(F.col("ingestion_datetime").desc(), completude.desc())
    return df.withColumn("_rn", F.row_number().over(janela)).filter(F.col("_rn") == 1).drop("_rn")


def numero_seguro(coluna, tipo="double"):
    """Conversão numérica segura: normaliza separadores e só faz cast se o texto for um número válido;
    qualquer outra coisa (texto deslocado, 'N/A', frases) vira NULL.
      '154,34'   -> 154.34   (vírgula decimal)
      '1.234,56' -> 1234.56  (milhar com ponto + decimal com vírgula)
    """
    c = F.trim(coluna.cast("string"))
    c = (F.when(c.rlike(r"^-?\d{1,3}(\.\d{3})+,\d+$"), F.regexp_replace(F.regexp_replace(c, r"\.", ""), ",", "."))
          .when(c.rlike(r"^-?\d+,\d+$"), F.regexp_replace(c, ",", "."))
          .otherwise(c))
    valido = c.rlike(r"^-?\d+(\.\d+)?$")
    if tipo == "int":
        return F.when(valido, c.cast("double").cast("int"))
    return F.when(valido, c.cast(tipo))

## 7 (executada primeiro). `silver.tb_cotacao_dolar`

Executada **antes** das demais por ser dependência do financeiro (conversão para BRL).

- Vários boletins por dia → fica o **último boletim** do dia (fechamento).
- Calendário contínuo (`sequence`) do primeiro ao último dia observado.
- **Forward fill**: sábados, domingos e feriados herdam a cotação do último dia útil disponível; a coluna `cotacao_preenchida` sinaliza esses dias.

In [0]:
cot = spark.table("bronze.tb_cotacao_dolar")

cot_dia = (
    cot
    .withColumn("ts_cotacao", F.to_timestamp(F.substring("dataHoraCotacao", 1, 19), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("data_cotacao", F.to_date("ts_cotacao"))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("decimal(10,4)"))
    .filter(F.col("data_cotacao").isNotNull() & (F.col("cotacao_compra") > 0))
)

# Último boletim de cada dia (e, em reprocessamentos, a ingestão mais recente)
w_dia = Window.partitionBy("data_cotacao").orderBy(F.col("ts_cotacao").desc(), F.col("ingestion_datetime").desc())
cot_dia = (cot_dia.withColumn("_rn", F.row_number().over(w_dia)).filter("_rn = 1")
                  .select("data_cotacao", "cotacao_compra"))

limites = cot_dia.agg(F.min("data_cotacao").alias("ini"), F.max("data_cotacao").alias("fim")).first()

# Calendário contínuo (todos os dias corridos, inclusive fins de semana e feriados)
calendario = spark.range(1).select(
    F.explode(F.sequence(F.lit(limites["ini"]), F.lit(limites["fim"]), F.expr("INTERVAL 1 DAY"))).alias("data_cotacao")
)

# Forward fill: último valor não nulo até a linha corrente
w_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
silver_cotacao = (
    calendario.join(cot_dia, "data_cotacao", "left")
    .withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())          # True = dia sem cotação original
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(w_ffill))
    .select("data_cotacao", "cotacao_compra", "cotacao_preenchida")
)

silver_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_cotacao_dolar")
display(spark.table("silver.tb_cotacao_dolar").orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra,cotacao_preenchida
2026-09-30,5.3601,false
2026-10-01,5.3904,false
2026-10-02,5.3577,false
2026-10-03,5.3577,true
2026-10-04,5.3577,true
2026-10-05,5.3885,false
2026-10-06,5.3651,false
2026-10-07,5.3613,false


## 1. `silver.tb_info_filmes`

Regras: normalizar `status` **antes** de traduzir; datas multi-formato; deduplicação pela ingestão mais recente; `ano_lancamento` derivado.

**Sobre datas ambíguas (`10/05/2018`):** perfilando a origem, no padrão com barra o 1º campo passa de 12 em 6.201 linhas e o 2º **nunca** passa de 12 ⇒ é `dd/MM/yyyy`. No padrão com hífen ocorre o inverso (2º campo > 12 em 2.770 linhas, 1º nunca) ⇒ é `MM-dd-yyyy`. Como fallback, se o formato principal falhar (ex.: mês 13), tentamos o formato alternativo; só é NULL quando **nenhum** dá certo.

In [0]:
# ---------- Normalização + tradução do status ----------
# 1) troca '-'/'_' por espaço  2) colapsa espaços  3) trim  4) minúsculas
status_norm = F.lower(F.trim(F.regexp_replace(F.regexp_replace(F.col("status"), r"[\-_]+", " "), r"\s+", " ")))

MAPA_STATUS = {
    "released":        "Lançado",
    "post production": "Pós-Produção",
    "in production":   "Em Produção",
    "planned":         "Planejado",
    "rumored":         "Rumores",
    "canceled":        "Cancelado",
    "cancelled":       "Cancelado",   # variação ortográfica britânica
}
status_traduzido = F.lit("Não Informado")           # corrompidos / não mapeáveis (ex.: status = '2016-10-25')
for original, traduzido in MAPA_STATUS.items():
    status_traduzido = F.when(status_norm == original, F.lit(traduzido)).otherwise(status_traduzido)

# ---------- Data multi-formato ----------
# Cada regra: (regex que identifica o padrão, [formatos a tentar em ordem])
REGRAS_DATA = [
    (r"^\d{4}-\d{2}-\d{2}$", ["yyyy-MM-dd"]),
    (r"^\d{4}/\d{2}/\d{2}$", ["yyyy/MM/dd"]),
    (r"^\d{2}/\d{2}/\d{4}$", ["dd/MM/yyyy", "MM/dd/yyyy"]),   # barra = dia/mês (validado no perfil)
    (r"^\d{2}-\d{2}-\d{4}$", ["MM-dd-yyyy", "dd-MM-yyyy"]),   # hífen = mês-dia (validado no perfil)
    (r"^\d{2}\.\d{2}\.\d{4}$", ["dd.MM.yyyy"]),
]
rd = F.trim(F.col("release_date"))
tentativas = []
for regex, formatos in REGRAS_DATA:
    parseadas = [F.to_date(rd, f) for f in formatos]
    tentativas.append(F.when(rd.rlike(regex), F.coalesce(*parseadas)))
data_lancamento = F.coalesce(*tentativas)             # nenhum padrão serviu => NULL

duracao = F.when(F.trim(F.col("runtime")).rlike(r"^\d+$"), F.trim(F.col("runtime")).cast("int"))

info = (
    spark.table("bronze.tb_movies_info")
    .filter(somente_numericos_validos("id"))
    .select(
        F.trim("id").alias("id_filme"),
        F.trim("title").alias("titulo"),
        F.trim("original_title").alias("titulo_original"),
        data_lancamento.alias("data_lancamento"),
        duracao.alias("duracao_minutos"),
        F.trim("original_language").alias("idioma_original"),
        status_traduzido.alias("status_filme"),
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("frase_divulgacao"),
        "ingestion_datetime",
    )
)

silver_info = (
    deduplicar_por_filme(info)                       # 1 linha por filme (versão mais recente)
    .withColumn("ano_lancamento", F.year("data_lancamento"))
    .drop("ingestion_datetime")
)

silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_info_filmes")

print("Linhas:", spark.table("silver.tb_info_filmes").count(),
      "| filmes únicos:", spark.table("silver.tb_info_filmes").select("id_filme").distinct().count())
display(spark.table("silver.tb_info_filmes").groupBy("status_filme").count().orderBy(F.desc("count")))
display(spark.table("silver.tb_info_filmes").filter("data_lancamento IS NULL").limit(10))

Linhas: 97611 | filmes únicos: 97611


status_filme,count
Lançado,96261
Pós-Produção,697
Em Produção,604
Planejado,47
Não Informado,2


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
482400,\Death,"Be Not Proud: The Making of \""\""The Exorcist III\""\""\""""",null,null,\Death,Não Informado,Released,null,null
543670,\Ding Dong,"You're Dead! The Making of \""\""House\""\""\""""",null,null,\Ding Dong,Não Informado,Released,null,null


## 2. `silver.tb_financeiro_filmes`

Regras: ausências textuais → NULL; remover `$`, `USD`, espaços, separador de milhar; interpretar sufixos `K/M/B` (`10.0K` = 10.000; `34.0M` = 34.000.000); `DECIMAL(18,2)`; `<= 0` → NULL; BRL pela cotação; lucro e margem.

**Decisão sobre NULLs na aritmética:** `lucro = receita − orçamento` só é calculado quando **os dois** valores existem. Em Spark, `NULL − x = NULL` (não gera erro nem quebra o pipeline), e assim evitamos fabricar lucro falso tratando orçamento ausente como zero. A margem usa `CASE WHEN receita > 0` para evitar divisão por zero.

**Cotação aplicada:** a mais recente disponível em `silver.tb_cotacao_dolar` (taxa única aplicada a todos os filmes, conforme enunciado "aplicando a taxa de cotação obtida").

In [0]:
TEXTOS_AUSENTES = ["unknown", "não informado", "nao informado", "n/a", "na", "null", "none", "-", ""]


def limpar_moeda(coluna):
    """Texto monetário sujo -> DECIMAL(18,2) em USD, ou NULL.
    Exemplos: '$ 10000' / 'USD 10000' -> 10000 | '10.0K' -> 10000 | '34.0M' -> 34000000 | 'Unknown' -> NULL
    """
    bruto = F.upper(F.trim(coluna.cast("string")))
    ausente = F.lower(F.trim(coluna.cast("string"))).isin(TEXTOS_AUSENTES) | coluna.isNull()

    sufixo = F.regexp_extract(bruto, r"([KMB])$", 1)                    # K, M, B ou ''
    numero = F.regexp_replace(bruto, r"[^0-9.,\-]", "")                 # tira $, USD, espaços e o sufixo
    numero = F.regexp_replace(numero, ",", "")                          # vírgula = milhar (padrão US)
    # '1.234.567' (ponto de milhar, sem sufixo) -> '1234567'
    numero = F.when((sufixo == "") & numero.rlike(r"^\d{1,3}(\.\d{3})+$"), F.regexp_replace(numero, r"\.", "")).otherwise(numero)

    multiplicador = (F.when(sufixo == "K", F.lit(1e3)).when(sufixo == "M", F.lit(1e6))
                      .when(sufixo == "B", F.lit(1e9)).otherwise(F.lit(1.0)))
    valor = F.when(~ausente & numero.rlike(r"^-?\d+(\.\d+)?$"),
                   (numero.cast("decimal(30,4)") * multiplicador.cast("decimal(30,4)")))
    valor = valor.cast("decimal(18,2)")
    return F.when(valor > 0, valor)                                      # zero/negativo => ausente


# Taxa de câmbio mais recente da silver de cotação
linha_taxa = spark.table("silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).first()
TAXA_USD_BRL = float(linha_taxa["cotacao_compra"])
print(f"Cotação aplicada: R$ {TAXA_USD_BRL:.4f} (data {linha_taxa['data_cotacao']})")

fin = (
    spark.table("bronze.tb_movies_financials")
    .filter(somente_numericos_validos("id"))
    .select(
        F.trim("id").alias("id_filme"),
        limpar_moeda(F.col("budget")).alias("orcamento_usd"),
        limpar_moeda(F.col("revenue")).alias("receita_usd"),
        "ingestion_datetime",
    )
)

silver_fin = (
    deduplicar_por_filme(fin).drop("ingestion_datetime")
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(TAXA_USD_BRL), 2).cast("decimal(18,2)"))
    .withColumn("receita_brl",   F.round(F.col("receita_usd")   * F.lit(TAXA_USD_BRL), 2).cast("decimal(18,2)"))
    # Lucro: NULL se faltar receita OU orçamento (NULL propaga; sem erro e sem lucro fictício)
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    # Margem % = lucro / receita * 100 ; protegida contra divisão por zero/NULL
    .withColumn("margem_lucro_pct",
                F.when(F.col("receita_usd") > 0, F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2)).cast("double"))
    .select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_pct")
)

silver_fin.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_financeiro_filmes")
display(spark.table("silver.tb_financeiro_filmes").orderBy(F.desc("receita_usd")).limit(10))

Cotação aplicada: R$ 5.3613 (data 2026-10-07)


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
299534,356000000.00,2800000000.00,1908622800.00,15011640000.00,2444000000.00,13103017200.00,87.29
76600,460000000.00,2320250281.00,2466198000.00,12439557831.53,1860250281.00,9973359831.53,80.17
299536,300000000.00,2052415039.00,1608390000.00,11003612748.59,1752415039.00,9395222748.59,85.38
634649,200000000.00,1921847111.00,1072260000.00,10303598916.20,1721847111.00,9231338916.20,89.59
420818,260000000.00,1663075401.00,1393938000.00,8916246147.38,1403075401.00,7522308147.38,84.37
361743,170000000.00,1488732821.00,911421000.00,7981543273.23,1318732821.00,7070122273.23,88.58
346698,145000000.00,1428545028.00,777388500.00,7658858458.62,1283545028.00,6881469958.62,89.85
502356,100000000.00,1355725263.00,536130000.00,7268449852.52,1255725263.00,6732319852.52,92.62
284054,200000000.00,1349926083.00,1072260000.00,7237358708.79,1149926083.00,6165098708.79,85.18
181808,200000000.00,1332698830.00,1072260000.00,7144998237.28,1132698830.00,6072738237.28,84.99


## 3. `silver.tb_metricas_engajamento`

- Popularidade: normaliza `154,34` → `154.34` antes do cast (sem virar NULL silenciosamente).
- *Column shift*: textos nas colunas numéricas viram NULL via `numero_seguro` (regex-guard), sem quebrar o job.
- Notas TMDB/IMDb fora de **0–10** (inclui erro de escala, ex.: `79.97`) → NULL. Votos/popularidade negativos → NULL.

In [0]:
def nota_valida(c):
    return F.when((c >= 0) & (c <= 10), c)            # fora da faixa (inclui escala x10) => NULL


def nao_negativo(c):
    return F.when(c >= 0, c)                          # negativo => NULL


met = (
    spark.table("bronze.tb_movies_metrics")
    .filter(somente_numericos_validos("id"))
    .select(
        F.trim("id").alias("id_filme"),
        nao_negativo(numero_seguro(F.col("popularity"), "double")).alias("popularidade"),
        nota_valida(numero_seguro(F.col("vote_average"), "double")).alias("nota_media_tmdb"),
        nao_negativo(numero_seguro(F.col("vote_count"), "int")).alias("qtd_votos_tmdb"),
        nota_valida(numero_seguro(F.col("averageRating"), "double")).alias("nota_media_imdb"),
        nao_negativo(numero_seguro(F.col("numVotes"), "int")).alias("qtd_votos_imdb"),
        "ingestion_datetime",
    )
)

silver_met = deduplicar_por_filme(met).drop("ingestion_datetime")
silver_met.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_metricas_engajamento")

display(spark.table("silver.tb_metricas_engajamento").describe())

summary,id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
count,95115,91579,91538,87741,83242,85310
mean,736592.0760763286,3.231584585986616,3.754855633725892,55.799512200681555,6.239276446985891,3068.812460438401
stddev,256369.17100048336,25.872476559045264,3.2962122981577506,530.0068371831587,1.4316499605267088,29216.699842949864
min,1000058,0.0,0.0,0,0.0,0
max,999999,2994.357,10.0,28894,10.0,1735566


## 4. `silver.tb_avaliacoes_usuarios`

- Duplicatas **integrais** (filme + usuário + nota + comentário) removidas.
- `nota_usuario` fora de 0–10 → NULL.
- Comentário nulo/vazio/só espaços → `"Sem comentário"`.

In [0]:
rev = (
    spark.table("bronze.tb_movies_reviews")
    .filter(somente_numericos_validos("id"))
    .select(
        F.trim("id").alias("id_filme"),
        F.trim("nome").alias("nome_usuario"),
        nota_valida(numero_seguro(F.col("nota"), "double")).alias("nota_usuario"),
        F.trim("comentario").alias("comentario_usuario"),
    )
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])   # duplicata integral
    .withColumn(
        "comentario_usuario",
        F.when(F.col("comentario_usuario").isNull() | (F.col("comentario_usuario") == ""), F.lit("Sem comentário"))
         .otherwise(F.col("comentario_usuario")),
    )
)

rev.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_avaliacoes_usuarios")
display(spark.table("silver.tb_avaliacoes_usuarios").limit(10))

id_filme,nome_usuario,nota_usuario,comentario_usuario
1100094,Gabriel Carvalho 581,6.2,"Aceitável, mas esperava mais."
628575,Alexandre Barbosa 220,0.3,Péssimo em todos os sentidos.
573249,Rodrigo Oliveira 273,0.5,Péssimo em todos os sentidos.
592539,Pedro Costa 181,4.8,"Não gostei, história confusa."
464493,Adriana Dias 257,0.4,Péssimo em todos os sentidos.
1199748,Eduardo Dias 177,6.0,"Poderia ser melhor, mas não é ruim."
1055332,Gustavo Dias 303,null,Sem comentário
599134,Larissa Lopes 330,2.6,Péssimo em todos os sentidos.
424011,Vinícius Ferreira 405,0.5,Não recomendo de jeito nenhum.
392834,Juliana Castro 595,8.8,Excepcional! História envolvente e atuações impecáveis.


## 5. `silver.tb_generos` e 6. `silver.tb_pessoas_empresas`

Ambas nascem de `bronze.tb_credits_and_tags`, deduplicada por filme (ingestão mais recente).

**Gêneros:** `;` e `|` são convertidos em `,` **antes** do `split`. Para eliminar os resíduos do *column shift* (notas `0.6`, caminhos `/abc.jpg`, nomes de países, frases) usamos uma **lista de domínio** (gêneros TMDB) — só o que pertence ao domínio passa; é a forma mais segura de garantir que "textos descritivos e valores numéricos deslocados" não entrem.

**Pessoas/empresas:** `split` + `explode` por tipo (`cast`→Ator, `directors`→Diretor, `writers`→Roteirista, `production_companies`→Produtora), com filtros: remove `N/A`, `[]`, `Unknown`, números puros (`6.6`), caminhos de imagem, frases longas (> 6 palavras para pessoas / > 8 para produtoras), keywords em minúsculo no início (`short film`, `anime`) para pessoas; padroniza capitalização com `initcap(lower(...))` e elimina duplicatas.

In [0]:
credits = deduplicar_por_filme(
    spark.table("bronze.tb_credits_and_tags")
    .filter(somente_numericos_validos("id"))
    .select(F.trim("id").alias("id_filme"), "genres", "production_companies", "directors", "writers", "cast",
            "ingestion_datetime")
)

# ---------------- 5. GÊNEROS ----------------
GENEROS_DOMINIO = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama", "Family",
                   "Fantasy", "History", "Horror", "Music", "Mystery", "Romance", "Science Fiction",
                   "TV Movie", "Thriller", "War", "Western"]
lookup_generos = spark.createDataFrame([(g.lower(), g) for g in GENEROS_DOMINIO], ["genero_lower", "genero"])

silver_generos = (
    credits
    .select("id_filme",
            # separadores heterogêneos (',' ';' '|') padronizados para ',' antes do split
            F.explode(F.split(F.regexp_replace(F.col("genres"), r"[;|]", ","), ",")).alias("genero_bruto"))
    .withColumn("genero_lower", F.lower(F.trim("genero_bruto")))
    .join(F.broadcast(lookup_generos), "genero_lower", "inner")       # só o que pertence ao domínio
    .select("id_filme", "genero")
    .distinct()
)
silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_generos")
display(spark.table("silver.tb_generos").groupBy("genero").count().orderBy(F.desc("count")))

genero,count
Drama,32615
Documentary,19234
Comedy,18827
Thriller,10400
Horror,9833
Romance,7717
Action,6114
Crime,4786
Animation,4524
TV Movie,4115


In [0]:
# ---------------- 6. PESSOAS / EMPRESAS ----------------
PLACEHOLDERS = ["n/a", "na", "none", "unknown", "null", "[]", "não informado", "nao informado", ""]


def explodir_entidades(base, coluna, tipo):
    """split+explode de uma coluna multivalorada, com limpeza de resíduos, devolvendo (id_filme, nome_entidade, tipo_entidade)."""
    separado = F.regexp_replace(F.col(coluna), r"[;|]", ",")                       # separadores extras
    df = base.select("id_filme", F.explode(F.split(separado, ",")).alias("nome_bruto"))

    nome = F.regexp_replace(F.col("nome_bruto"), r"[\[\]\"\\]", "")                # colchetes, aspas, barra invertida
    nome = F.regexp_replace(nome, r"^['\s]+|['\s]+$", "")                          # aspas simples nas pontas (preserva O'Brien)
    nome = F.trim(F.regexp_replace(nome, r"\s+", " "))                             # espaços duplicados
    df = df.withColumn("nome", nome)

    n_palavras = F.size(F.split(F.col("nome"), " "))
    valido = (
        ~F.lower(F.col("nome")).isin(PLACEHOLDERS)                                 # N/A, [], Unknown...
        & ~F.col("nome").rlike(r"^[\d\s.,\-]+$")                                   # números deslocados: 6.6, 0.0286
        & ~F.col("nome").rlike(r"(?i)^/|\.jpg|\.png|https?:")                      # caminhos de imagem / URLs
        & (F.length("nome") <= 80)                                                 # frases descritivas longas
    )
    if tipo == "Produtora":
        valido = valido & (n_palavras <= 8)
    else:  # pessoas: nomes curtos e iniciados por maiúscula (descarta keywords: 'short film', 'anime')
        valido = valido & (n_palavras <= 6) & ~F.col("nome").rlike(r"^[a-z]")

    return (df.filter(valido)
              .select("id_filme",
                      F.initcap(F.lower("nome")).alias("nome_entidade"),           # padroniza capitalização
                      F.lit(tipo).alias("tipo_entidade")))


MAPA_ENTIDADES = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista", "production_companies": "Produtora"}

silver_pessoas = None
for coluna, tipo in MAPA_ENTIDADES.items():
    parte = explodir_entidades(credits, coluna, tipo)
    silver_pessoas = parte if silver_pessoas is None else silver_pessoas.unionByName(parte)

silver_pessoas = silver_pessoas.distinct()      # elimina duplicatas (mesmo filme + nome + tipo)
silver_pessoas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_pessoas_empresas")

display(spark.table("silver.tb_pessoas_empresas").groupBy("tipo_entidade").agg(
    F.count("*").alias("registros"), F.countDistinct("nome_entidade").alias("entidades_distintas")))


tipo_entidade,registros,entidades_distintas
Ator,543103,268827
Diretor,99991,63304
Roteirista,125086,84468
Produtora,119698,45037


## Validação final da Silver

In [0]:
for t in ["tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento", "tb_avaliacoes_usuarios",
          "tb_generos", "tb_pessoas_empresas", "tb_cotacao_dolar"]:
    df = spark.table(f"silver.{t}")
    print(f"silver.{t:28s} linhas={df.count():>9,}")

# Unicidade por filme nas tabelas de grão "filme"
for t in ["tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento"]:
    df = spark.table(f"silver.{t}")
    dup = df.groupBy("id_filme").count().filter("count > 1").count()
    print(f"{t:28s} ids duplicados: {dup}")

silver.tb_info_filmes               linhas=   97,611
silver.tb_financeiro_filmes         linhas=   99,006
silver.tb_metricas_engajamento      linhas=   95,115
silver.tb_avaliacoes_usuarios       linhas=   32,412
silver.tb_generos                   linhas=  141,939
silver.tb_pessoas_empresas          linhas=  887,878
silver.tb_cotacao_dolar             linhas=        8
tb_info_filmes               ids duplicados: 0
tb_financeiro_filmes         ids duplicados: 0
tb_metricas_engajamento      ids duplicados: 0
